In [ ]:
# GEO query and DEG analysis script

# Install dependencies if not already installed:
!pip install GEOparse pandas matplotlib seaborn scipy numpy

import GEOparse
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns# -----------------------------
# Step 1: Download GEO dataset
# -----------------------------
gse_id = "GSE103174"  # <-- Change GEO Series
gse = GEOparse.get_GEO(geo=gse_id, destdir=".")

print(f"Dataset: {gse_id}")
print(f"Title: {gse.metadata['title'][0]}")
print(f"Samples: {len(gse.gsms)}")# Step 2: Extract Expression Matrix
# -----------------------------
dfs = []
for gsm_name, gsm in gse.gsms.items():
    if gsm.table is not None and "VALUE" in gsm.table.columns:
        df = gsm.table[["ID_REF", "VALUE"]].copy()
        df.rename(columns={"VALUE": gsm_name}, inplace=True)
        dfs.append(df)

expr_df = dfs[0]
for df in dfs[1:]:
    expr_df = expr_df.merge(df, on="ID_REF")

expr_df = expr_df.drop_duplicates(subset="ID_REF")
expr_df.set_index("ID_REF", inplace=True)

print("Expression Matrix Shape:", expr_df.shape)

# Save raw matrix
expr_df.to_csv("expression.csv")
print("✅ Saved: expression.csv")
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

n = len(expr_df.columns)
control = expr_df.iloc[:, :n//2]
treated = expr_df.iloc[:, n//2:]

pvals, logFCs, t_stats, avg_exp_values = [], [], [], []

for gene in expr_df.index:
    control_values = control.loc[gene]
    treated_values = treated.loc[gene]

    if control_values.dropna().empty or treated_values.dropna().empty:
        stat, p, logFC, avg_exp = np.nan, np.nan, np.nan, np.nan
    else:
        stat, p = ttest_ind(control_values, treated_values, nan_policy="omit")
        logFC = treated_values.mean() - control_values.mean()
        avg_exp = expr_df.loc[gene].mean()

    t_stats.append(stat)
    pvals.append(p)
    logFCs.append(logFC)
    avg_exp_values.append(avg_exp)

deg_df = pd.DataFrame({
    "Gene": expr_df.index,
    "logFC": logFCs,
    "t": t_stats,
    "Pvalue": pvals,
    "AvgExp": avg_exp_values
})

# Safe log10
deg_df["-log10p"] = -np.log10(deg_df["Pvalue"].replace(0, np.nan))

# Adjust p-values
valid = deg_df["Pvalue"].notna()
if valid.sum() > 1:
    deg_df.loc[valid, "adjPvalue"] = multipletests(deg_df.loc[valid, "Pvalue"], method='fdr_bh')[1]
else:
    deg_df["adjPvalue"] = np.nan

deg_df.to_csv("DEG.csv", index=False)
print("✅ Saved: DEG.csv")

13-Apr-2026 03:21:32 DEBUG utils - Directory . already exists. Skipping.
DEBUG:GEOparse:Directory . already exists. Skipping.
13-Apr-2026 03:21:32 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE103nnn/GSE103174/soft/GSE103174_family.soft.gz to ./GSE103174_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE103nnn/GSE103174/soft/GSE103174_family.soft.gz to ./GSE103174_family.soft.gz
100%|██████████| 29.3M/29.3M [00:00<00:00, 78.1MB/s]
13-Apr-2026 03:21:33 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
13-Apr-2026 03:21:33 DEBUG downloader - Moving /tmp/tmpasu8yknh to /content/GSE103174_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmpasu8yknh to /content/GSE103174_family.soft.gz
13-Apr-2026 03:21:33 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE103nnn/GSE103174/soft/GSE103174_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE

Dataset: GSE103174
Title: Expression data from lung tissue in mild-moderate COPD
Samples: 53
Expression Matrix Shape: (15040, 53)
✅ Saved: expression.csv
✅ Saved: DEG.csv
